In [1]:
from Data_query.trino_config import *
import numpy as np
from visualisation import *
import pytz

In [10]:
stop_trino()

Trino service stopping triggered.


In [8]:
sleep(40)

In [2]:
big_workers = 0
workers = 1
num_workers = max(workers, big_workers)
ensure_trino_running(worker_desired_count = workers, big_worker_desired_count=big_workers)
# sleep(40)

Trino service is not running. Starting the service...
Trino service triggered.
Service trino-service is now stable.


In [3]:

num_parts=1
time_bin_interval = '5' # in minutes
def run_func(args):
    year, month, part = args
    # time_filter = f"year = {year} and month = {month}"
    time_filter = f"year = {year}"
    site_filters = f"site_id in (1567203727, 250484079, 1792599725, 696192939)"
    df = iceberg_sql(f"""
                    with 
                     test_data as (
                        SELECT
                            site_id,
                            actual_day,
                            cs_day,
                            t_stamp,
                            P_kw_norm,
                            P_kw_norm_cs,
                            S_norm,
                            V,
                            CAST(
                                date_trunc('minute', t_stamp + interval '10' hour)
                                - interval '1' minute * (minute(t_stamp + interval '10' hour) % {time_bin_interval})
                                AS TIME) AS tod_bin,
                            GHI/GHI_cs AS x,
                            GHI_cs,
                            P_kw_norm/ NULLIF(P_kw_norm_cs, 0.0) AS y
                        FROM structured_data
                        WHERE P_kw_norm_cs > 0.2 AND GHI > 50 and P_kw_norm > 0.05 and P_kw_norm <= P_kw_norm_cs
                             and {time_filter} and {site_filters} 
                    ),
                    validation_on_test_data AS (
                        select 
                            t.site_id, 
                            t.t_stamp, 
                            actual_day, 
                            cs_day,
                            x as GHI_norm, 
                            GHI_cs,
                            P_kw_norm, 
                            P_kw_norm_cs,
                            P_kw_norm_cs * (a + b * x) AS P_kw_norm_est,
                            V,
                            S_norm
                        from test_data t 
                            join single_site_pv_ghi_model m on t.site_id = m.site_id and t.tod_bin = m.tod_bin
                    )
                    SELECT *
                    FROM validation_on_test_data
                        """)

                    # 

    # sleep(20)
    # print(f"Completed {time_filter}, {split_cons.replace('system.bucket(postcode, 16)', 'bucket')}, part {part}, count: {df['site_id'].nunique()}")
    return df
tasks = [(year, month, part) for year in (2024, ) for month in range(1, 2) 
         for part in range(0, num_parts)]
        #   for split_cons in ['system.bucket(postcode, 16) > -1'] ]
            
try:         
    res_test = trino_parallel(run_func, tasks, num_workers=1)
except Exception as e:
    print(f"Error during data retrieval: {e}")
finally:
    # stop_trino()
    pass
res_test['t_stamp'] = pd.to_datetime(res_test['t_stamp']).dt.tz_localize('utc').dt.tz_convert(pytz.FixedOffset(10*60))
res_test['GHI_norm'] = res_test['GHI_norm'].fillna(-1)
res_test

Combining results from all tasks.


,site_id,t_stamp,actual_day,cs_day,GHI_norm,GHI_cs,P_kw_norm,P_kw_norm_cs,P_kw_norm_est,V,S_norm
0,696192939,2024-02-01 10:15:00+10:00,2024-02-01,2024-02-02,0.752945,960.88,0.510438,0.796984,0.638701,237.066667,0.510441
1,696192939,2024-02-01 10:20:00+10:00,2024-02-01,2024-02-02,1.010823,960.88,0.680589,0.800033,0.806507,238.400000,0.680594
2,696192939,2024-02-01 10:25:00+10:00,2024-02-01,2024-02-02,0.993616,977.52,0.737206,0.803891,0.799830,238.183333,0.737211
3,696192939,2024-02-01 10:40:00+10:00,2024-02-01,2024-02-02,0.826713,992.34,0.739835,0.818790,0.694360,236.733333,0.739838
4,696192939,2024-02-01 10:45:00+10:00,2024-02-01,2024-02-02,0.816047,1005.31,0.539021,0.819271,0.679075,236.233333,0.539024
...,...,...,...,...,...,...,...,...,...,...,...
77326,1567203727,2024-07-24 16:30:00+10:00,2024-07-24,2024-07-18,0.844372,241.73,0.159193,0.259662,0.271700,247.300000,0.170972
77327,1567203727,2024-07-24 16:35:00+10:00,2024-07-24,2024-07-18,1.002702,203.56,0.144891,0.248449,0.248438,247.500000,0.158174
77328,1567203727,2024-07-24 16:40:00+10:00,2024-07-24,2024-07-18,0.444783,203.56,0.226355,0.236074,0.243472,248.000000,0.233507
77329,1567203727,2024-07-24 16:45:00+10:00,2024-07-24,2024-07-18,0.547500,165.37,0.211450,0.225030,0.228875,247.800000,0.219336


In [ ]:
df0.query(f"actual_day=='2024-01-03'")

,site_id,t_stamp,actual_day,cs_day,GHI,GHI_cs,P_kw_norm,P_kw_norm_cs,P_kw_norm_est,V,S_norm


In [ ]:
# sample_site_id = 1567203727
# sample_site_id = 250484079
# sample_site_id = 1792599725
# sample_site_id = 696192939
df0 = res_test.query(f"site_id=={sample_site_id}").reset_index(drop=True)
t0 = df0['t_stamp'].min()
t1 = df0['t_stamp'].max()
t0 = t0 + pd.Timedelta(days=38)
t1 = t0 + pd.Timedelta(days=18)
# start_time = t0.strftime('%Y-%m-%d %H:%M:%S%z')
# end_time = t1.strftime('%Y-%m-%d %H:%M:%S%z')
if t0.time() == pd.Timestamp('00:00:00').time():
    start_time = t0.strftime('%Y-%m-%d %H:%M:%S%z')
else:
    start_time = (t0 + pd.Timedelta(days=1)).replace(hour=0, minute=0, second=0).strftime('%Y-%m-%d %H:%M:%S%z')
if t1.time() == pd.Timestamp('00:00:00').time():
    end_time = t1.strftime('%Y-%m-%d %H:%M:%S%z')
else:
    end_time = t1.replace(hour=0, minute=0, second=0).strftime('%Y-%m-%d %H:%M:%S%z')

num_ticks = 24*2+1
save_as = f'Figures/GHInorm_{sample_site_id}.jpeg'
x_label = 'time'
y_labels = ['GHI_norm', 'Voltage (V)', 'Active Power (kW)', 'Active Power (kW)', 'Active Power (kW)']
plt_config = {
    'GHI_norm': [0, 0, '-', None, None], 'V': [0, 1, '-', None, None],
    'P_kw_norm': [1, 0, '-', None, None],'P_kw_norm_est': [1, 0, '-', None, None],'P_kw_norm_cs': [1, 0, '-', None, None]
}

color_nights=False
# color_by = 'group'
color_by = 'attribute'
ax_digit = '1.1f'
a=my_plot4(start_time, end_time, df0, plt_config=plt_config, ax_digit= ax_digit,
          group_attr='site_id', time_attr='t_stamp', color_nights=color_nights,cmap='plasma',
          figsize=[16/2.54,1.3],  same_scale=1, fontsize=5, fontname='DejaVu Sans', plot_total=False, plot_total_func=['sum', [lambda x: max(x), 'max']], 
          num_ticks=num_ticks, num_yticks=10, dpi=200,  x_format= '%H:%M', 
           legend_loc=['upper left', 'upper right', 'center left','upper left', 'lower left'], x_label=x_label, y_labels=y_labels, color_by=color_by,
plot_period=np.timedelta64(1, 'D'), save_as=save_as, rotation = 60, step=0, gridwidth=[0.2, .2], legend_join='-', title='', 
legend_i=0, title_i=0, only1title=0, onlyntime=0, show=False)
a.do()

saved as:  /home/hossein/CICCADA/Figures/GHInorm_1792599725.jpeg


In [6]:
df0

,site_id,t_stamp,actual_day,cs_day,GHI_norm,GHI_cs,P_kw_norm,P_kw_norm_cs,P_kw_norm_est,V,S_norm
0,1792599725,2024-09-01 10:00:00+10:00,2024-09-01,2024-09-01,1.000000,710.53,0.818349,0.821510,0.821510,253.20,0.917704
1,1792599725,2024-09-01 10:05:00+10:00,2024-09-01,2024-09-01,0.972423,730.68,0.821510,0.822288,0.799587,253.35,0.922273
2,1792599725,2024-09-01 10:10:00+10:00,2024-09-01,2024-09-01,1.000000,730.68,0.822288,0.827832,0.827832,253.35,0.922779
3,1792599725,2024-09-01 10:15:00+10:00,2024-09-01,2024-09-01,0.975358,749.14,0.827832,0.830739,0.809832,253.45,0.930534
4,1792599725,2024-09-01 10:20:00+10:00,2024-09-01,2024-09-01,1.000000,749.14,0.830739,0.832836,0.832836,253.70,0.935547
...,...,...,...,...,...,...,...,...,...,...,...
31038,1792599725,2024-02-01 09:25:00+10:00,2024-02-01,2024-02-02,0.422955,859.43,0.544136,0.896139,0.355554,252.05,0.662695
31039,1792599725,2024-02-01 09:30:00+10:00,2024-02-01,2024-02-02,0.539148,859.43,0.336517,0.899248,0.473517,251.20,0.479123
31040,1792599725,2024-02-01 09:35:00+10:00,2024-02-01,2024-02-02,0.507464,913.09,0.550580,0.912113,0.449218,252.20,0.665559
31041,1792599725,2024-02-01 09:50:00+10:00,2024-02-01,2024-02-02,0.680820,913.09,0.543315,0.912113,0.632588,253.70,0.687215
